In [0]:
%sql
create catalog cdf;
create schema cdf.silver_demo;


In [0]:
%sql
CREATE TABLE cdf.silver_demo.silver_customers (
  id INT,
  name STRING,
  city STRING
)
TBLPROPERTIES (delta.enableChangeDataFeed = true);


In [0]:
%sql
INSERT INTO cdf.silver_demo.silver_customers VALUES
(1,'Alice','NY'),
(2,'Bob','LA'),
(3,'Charlie','TX');


In [0]:
%sql
select * from cdf.silver_demo.silver_customers

In [0]:
%sql
describe history cdf.silver_demo.silver_customers

In [0]:
%sql
SELECT *
FROM table_changes('cdf.silver_demo.silver_customers', 1);

In [0]:
%sql
CREATE TABLE cdf.silver_demo.gold_customers (
  id INT,
  name STRING,
  city STRING
);


In [0]:
%sql
insert into cdf.silver_demo.gold_customers
select * from cdf.silver_demo.silver_customers

In [0]:
%sql
-- update
UPDATE cdf.silver_demo.silver_customers
SET city = 'Chicago'
WHERE id = 1;

-- delete
DELETE FROM cdf.silver_demo.silver_customers
WHERE id = 2;

-- insert
INSERT INTO cdf.silver_demo.silver_customers VALUES
(4,'David','Boston');


In [0]:
%sql
select * from table_changes("cdf.silver_demo.silver_customers",1)

In [0]:
df=spark.sql("""select * from table_changes("cdf.silver_demo.silver_customers",1)""")

In [0]:
display(df)

In [0]:
from delta.tables import DeltaTable

In [0]:
gold=DeltaTable.forName(spark,"cdf.silver_demo.gold_customers")

gold.alias("t").merge(df.alias("s"),"t.id=s.id").\
    whenMatchedDelete(condition="s._change_type='delete'").\
    whenMatchedUpdate(condition="s._change_type='update_postimage'",set={
        "id":"s.id",
        "name":"s.name",
        "city":"s.city"
    }).\
    whenNotMatchedInsert(condition="s._change_type='insert'",values={
        "id":"s.id",
        "name":"s.name",
        "city":"s.city"
        }).execute()

In [0]:
%sql
select * from cdf.silver_demo.gold_customers